# 2) LangChain Templates
# pip install -U google-genai langchain-core

# PromptTemplate becomes genuinely useful when the 
#   > Prompt Structure remains fixed 
#   > Values such as topic, audience, tone, and output_format change repeatedly.

In [ ]:
import os,warnings
from google import genai
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
import random
warnings.filterwarnings("ignore")

In [ ]:
env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# Create the Gemini client
client = genai.Client(api_key=gemini_key)

In [ ]:
def run(prompt):
    response = client.interactions.create(model="gemini-3.1-flash-lite", input=prompt)
    return response.output_text

In [ ]:
# Sinle input
# -----------
topic = "nature trek and adventure"

try:
    pt = PromptTemplate(input_variables=["topic"], 
                        template="what are the top 5 destinations for people who love {topic}?" )

    prompt = pt.format(topic=topic)
    
    print(prompt)
    response = run(prompt)
    print(response.content)

except Exception as e:
    print("Exception : " + str(e))

In [ ]:
# 2 Inputs
# --------
temp = "tell me a {p1} fact about {p2}"
pt = PromptTemplate.from_template(temp)
prompt = pt.format(p1="biological",p2="bones")
print(prompt)
run(prompt)

In [ ]:
# 3 Inputs
# ---------
temp = "give the list of top 5 {p1} in the city {p2} that is famous for its {p3}"
pt = PromptTemplate.from_template(temp)
prompt = pt.format(p1="restaurants",p2="Hyderabad",p3="diverse cuisine")
print(prompt)
response = run(prompt)

In [ ]:
# Generic function that can take any number of parameters
# -------------------------------------------------------

def RunTemplate(template,**topic):
    try:
        pt = PromptTemplate(template=template)
        prompt = pt.format(**topic)

        print(f"Generated Prompt:\n{prompt}")
        response = run(prompt)
        print(f"\nGemini Response:\n{response}")

    except Exception as e:
        print("Exception:", str(e))

In [ ]:
# ----------------------
# Run this one at a time
# ----------------------

# Single Placeholder
# topic = "nature trek and adventure"
# template = "What are the top 5 destinations for people who love {topic}"
# RunTemplate(template=template, topic=topic)

# # 2  Placeholders
# health_factor = ['less salt', 'excess sugar', 'too much exercise', 'less sleep']
# age_group = ['children', 'young adults', 'adults', 'senior citizens']
# template = "How does {health_factor} affect {age_group}?"
# RunTemplate(template=template, health_factor=random.choice(health_factor), age_group=random.choice(age_group))

# # 3 Placeholders
# top = [1,2,3,4,5]
# city = ['bangalore', 'pune', 'mumbai', 'chennai']
# place = ['restaurants', 'temples', 'rivers', 'beaches']
# template = "Give the list of top {top} in the city {city} that is famous for its {place}"
# RunTemplate(template=template, top=random.choice(top), city=random.choice(city), place=random.choice(place))

In [ ]:
def run_sports_template(sport, audience, tone, output_format):
    try:
        template = """
        You are a professional sports trainer.
        Explain the basic rules and essential skills of {sport}.
        Target audience: {audience}
        Tone: {tone}
        Output format: {output_format}

        Also include one practical training tip.
        """

        pt = PromptTemplate.from_template(template)

        prompt = pt.format(sport=sport, audience=audience, tone=tone, output_format=output_format)
        print("Generated Prompt:")
        print(prompt)

        return prompt

    except Exception as e:
        print("Exception:", str(e))
        return None

In [ ]:
run_sports_template(sport="Cricket", audience="Children between 10 and 14 years", tone="Simple and encouraging", output_format="Five bullet points")

In [ ]:
run_sports_template(sport="Soccer", audience="Girls between 10 and 14 years", tone="Encouraging", output_format="Five numbered points")

In [ ]:
# ---------
# Method 2
# --------

# py -m pip install langchain_google_genai
# python -m pip install --upgrade langchain-core langchain-google-genai google-genai

# ChatGoogleGenerativeAI: It allows a LangChain application to communicate with Google’s Gemini models.

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate


model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key=gemini_key)

In [ ]:
template = PromptTemplate(template="Write an article on the topic of {topic}")

prompt = template.invoke({'topic': 'AI'})
# prompt2 = template.invoke({'topic': 'Global Warming'})
response = model.invoke(prompt)
print(response.text)

# deeper prompting patterns

In [ ]:
import os
# from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os,warnings
from dotenv import load_dotenv

In [ ]:
env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
# Securely enter the Gemini API key
# os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

# Create the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key,temperature=0)

parser = StrOutputParser()

In [ ]:
# In-code exemplars - Few Shot Learning
# Few-shot prompting gives the model some input-output examples before presenting the actual input.

few_shot_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Classify customer feedback into one of these categories:
        Product, Delivery, Service, or Payment.

        Follow the examples:

        Feedback: "The mobile application crashes frequently."
        Category: Product

        Feedback: "My order arrived three days late."
        Category: Delivery

        Feedback: "The support executive was very helpful."
        Category: Service

        Feedback: "My card was charged twice."
        Category: Payment

        Return only the category.
        """
    ),
    ("human", "Feedback: {feedback}")
])

few_shot_chain = few_shot_prompt | llm | parser

result = few_shot_chain.invoke({"feedback": "The payment was deducted, but the order was not confirmed."})

print(result)

In [ ]:
# Instructional Scaffolding — Step-by-Step Guidance
# Instructional scaffolding breaks a complex task into explicit processing stages.

In [ ]:
scaffolding_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a customer complaint analyst.

        Analyse the complaint using these stages:

        1. Identify the main problem.
        2. Identify the likely business impact.
        3. Assign a priority: Low, Medium, or High.
        4. Recommend one immediate action.
        5. Present the final answer using the headings:
           - Problem
           - Business Impact
           - Priority
           - Recommended Action

        Keep the complete answer under 120 words.
        """
    ),
    ("human", "Customer complaint: {complaint}")
])

scaffolding_chain = scaffolding_prompt | llm | parser

result = scaffolding_chain.invoke({
    "complaint": """
    I transferred ₹25,000 through mobile banking.
    The amount was deducted, but the beneficiary has not received it.
    """
})

print(result)

In [ ]:
# Contextual Priming with Constraints and Roles
# Contextual priming tells Gemini who it should act as, provides relevant background, and defines operating constraints.

In [ ]:
context_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Role:
        You are a cybersecurity awareness trainer for a bank.

        Context:
        Your audience consists of non-technical bank employees who regularly
        use email, customer-management systems, and online banking portals.

        Constraints:
        - Use simple, non-technical language.
        - Give exactly three recommendations.
        - Each recommendation must be one sentence.
        - Focus only on phishing prevention.
        - Do not discuss malware, firewalls, or programming.
        - Do not use fear-based language.
        """
    ),
    ("human", "Create employee guidance for this situation: {situation}")
])

context_chain = context_prompt | llm | parser

result = context_chain.invoke({
    "situation": """
    An employee receives an urgent email asking them to verify a customer account by clicking a link.
    """
})

print(result)

In [ ]:
# Pattern Seeding and Iterative Refinement
# Here, the first prompt generates a broad draft. Later prompts progressively add constraints and improve it.

In [ ]:
# Stage 1: Generate a broad draft
broad_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a corporate training designer."
    ),
    (
        "human",
        """
        Create a broad outline for a training session on {topic}.
        Audience: {audience}
        """
    )
])

broad_chain = broad_prompt | llm | parser

draft_1 = broad_chain.invoke({"topic": "Responsible use of generative AI", "audience": "Bank employees"})

print("BROAD DRAFT\n")
print(draft_1)

In [ ]:
refinement_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You improve training outlines while preserving their useful content.
        """
    ),
    (
        "human",
        """
        Refine the following training outline.

        Existing outline:
        {draft}

        Add these constraints:
        - Total duration must be 60 minutes.
        - Divide it into four sections.
        - Include one practical activity.
        - Include the duration of every section.
        - Focus on data privacy, accuracy and human review.
        """
    )
])

refinement_chain = refinement_prompt | llm | parser

draft_2 = refinement_chain.invoke({"draft": draft_1})

print("\nREFINED DRAFT\n")
print(draft_2)

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You produce concise, classroom-ready training plans."
    ),
    (
        "human",
        """
        Improve the training plan below.

        Current plan:
        {draft}

        Additional constraints:
        - Add one learning objective for each section.
        - Add three assessment questions.
        - Use a professional tone.
        - Keep the complete response under 400 words.
        """
    )
])

final_chain = final_prompt | llm | parser

final_plan = final_chain.invoke({"draft": draft_2})

print("\nFINAL PLAN\n")
print(final_plan)

In [ ]:
# Negative Prompting
# Negative prompting explicitly states what should not appear in the response.

In [ ]:
negative_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an employee wellness assistant.

        Suggest practical ways to improve sleep quality.

        Include:
        - Safe, general lifestyle suggestions
        - Actions that an office employee can realistically follow
        - A short explanation for every suggestion

        Exclude:
        - Medicines or supplements
        - Medical diagnosis
        - Extreme diets
        - Expensive products
        - Unverified health claims
        - Suggestions requiring special equipment

        Give exactly five suggestions.
        """
    ),
    ("human", "User situation: {situation}")
])

negative_chain = negative_prompt | llm | parser

result = negative_chain.invoke({
    "situation": """
    I work at a computer throughout the day and have difficulty
    falling asleep at night.
    """
})

print(result)

# Selectors in Prompts

In [ ]:
# FixedExampleSelector
# The input examples are always the same, regardless of the user’s input.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [ ]:
env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")
	
# Securely enter the Gemini API key
# os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

# Create the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=gemini_key,temperature=0)

parser = StrOutputParser()

In [ ]:
# Fixed examples
examples = [
    {
        "feedback": "The mobile application crashes frequently.",
        "category": "Product"
    },
    {
        "feedback": "My order arrived three days late.",
        "category": "Delivery"
    },
    {
        "feedback": "The support executive was very helpful.",
        "category": "Service"
    },
    {
        "feedback": "My card was charged twice.",
        "category": "Payment"
    }
]


# Format for each example
example_prompt = PromptTemplate.from_template("Feedback: {feedback}\nCategory: {category}")

# Pass the fixed examples directly to FewShotPromptTemplate
prompt = FewShotPromptTemplate(examples=examples, example_prompt=example_prompt,
    prefix="""
    Classify customer feedback into one category:
    Product, Delivery, Service, or Payment.

    Follow these examples:
    """,
    suffix="""
    Feedback: {user_feedback}
    Category:
    """,
    input_variables=["user_feedback"]
)

In [ ]:
# Create the chain
chain = prompt | llm | StrOutputParser()

# Run the chain
result = chain.invoke({"user_feedback": "The amount was deducted, but the payment failed."})

print(result)

In [ ]:
# LengthBasedExampleSelector
# includes examples until the specified length limit is reached. If the user input is long, fewer examples are selected.

'''
If examples are short, more examples are selected.
If examples are long, fewer examples are selected.
A longer user input leaves less room for examples.
Selection is based on length, not Dee relevance.
It generally considers examples in their given order.
'''

In [ ]:
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.output_parsers import StrOutputParser

# Available examples
examples = [
    {
        "feedback": "The mobile application crashes.",
        "category": "Product"
    },
    {
        "feedback": "My order arrived late.",
        "category": "Delivery"
    },
    {
        "feedback": "The support executive was helpful.",
        "category": "Service"
    },
    {
        "feedback": "My card was charged twice.",
        "category": "Payment"
    }
]

# Define how each example should appear
example_prompt = PromptTemplate.from_template("Feedback: {feedback}\nCategory: {category}")

# Create the length-based selector
example_selector = LengthBasedExampleSelector(examples=examples, example_prompt=example_prompt, max_length=25)

# Create the few-shot prompt
prompt = FewShotPromptTemplate(example_selector=example_selector, example_prompt=example_prompt,
    prefix="""
Classify the feedback into one category:
Product, Delivery, Service, or Payment.

Examples:
""",
    suffix="""
Feedback: {input}
Category:
""",
    input_variables=["input"]
)

# Create the chain
chain = prompt | llm | StrOutputParser()

# Run the chain
user_input = "The amount was deducted, but the transaction failed."

result = chain.invoke({"input": user_input})

print(result)

# SemanticSimilarityExampleSelector

In [8]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [10]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=gemini_key)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [13]:
example_prompt = PromptTemplate.from_template("Feedback: {feedback}\nCategory: {category}")

In [14]:
selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    InMemoryVectorStore,
    k=2,
    input_keys=["feedback"]
)

In [15]:
# Few-shot prompt
prompt = FewShotPromptTemplate(example_selector=selector, example_prompt=example_prompt,
    prefix="""
    Classify the customer feedback as:
    Product, Delivery, Service, or Payment.

    Relevant examples:
    """,
suffix="""
    Feedback: {feedback}
    Category:
    """,
    input_variables=["feedback"]
)

In [16]:
chain = prompt | llm | StrOutputParser()

user_feedback = "The amount was deducted twice from my account."

# The invocation key is also "feedback"
result = chain.invoke({"feedback": user_feedback})
print("Category:", result)

Category: Category: Payment
